## Tópicos en reviews (Ollama) → Neon

Lee `reviews` (campo `texto_limpio`), clasifica tópicos con el mismo flujo que `topicos_reviews_ollama.ipynb`, y persiste en `review_topicos`. Los nombres de tópicos candidatos salen de la tabla `topicos`; si el modelo propone uno nuevo, se inserta en `topicos` (tipo `adicional`).

**Requisitos:** `DATABASE_URL` (PostgreSQL/Neon), [Ollama](https://ollama.com) con el modelo configurado (p. ej. `llama3`), `pip install ollama pandas psycopg2-binary tqdm`.

In [ ]:
import json
import os
import re
import unicodedata
from pathlib import Path

import pandas as pd
import psycopg2

try:
    import ollama
except ImportError:
    raise ImportError("Instala: pip install ollama pandas psycopg2-binary")

MODELO = "llama3"
MODEL_VERSION = f"ollama:{MODELO}"

DATABASE_URL = ""
# Procesamiento: None = todas las filas elegibles
N_MAX = 100
OFFSET = 0
# Si True, solo reviews sin ninguna fila en review_topicos
ONLY_WITHOUT_TOPICOS = True

print("Modelo:", MODELO, "| version BD:", MODEL_VERSION)
print("N_MAX:", N_MAX, "OFFSET:", OFFSET, "ONLY_WITHOUT_TOPICOS:", ONLY_WITHOUT_TOPICOS)

Modelo: llama3 | version BD: ollama:llama3
N_MAX: 100 OFFSET: 0 ONLY_WITHOUT_TOPICOS: True


In [4]:
def _norm_nombre(s: str) -> str:
    s = (s or "").strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s


def slugify(text: str, max_len: int = 60) -> str:
    s = unicodedata.normalize("NFKD", text or "")
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = s.lower()
    s = re.sub(r"[^a-z0-9]+", "_", s).strip("_")
    return (s[:max_len] if s else "topico")[:max_len]


def conectar():
    return psycopg2.connect(DATABASE_URL)


def cargar_topicos_desde_bd(conn) -> tuple[list[str], dict[str, int]]:
    """Lista de nombres (para el prompt) y mapa nombre_normalizado -> id."""
    with conn.cursor() as cur:
        cur.execute(
            """
            SELECT id, nombre FROM topicos
            WHERE activo IS DISTINCT FROM FALSE
            ORDER BY id
            """
        )
        rows = cur.fetchall()
    nombres = [r[1] for r in rows]
    por_norm = {_norm_nombre(r[1]): r[0] for r in rows}
    return nombres, por_norm


conn = conectar()
try:
    TOPICOS_NOMBRES, TOPICO_POR_NOMBRE_NORM = cargar_topicos_desde_bd(conn)
finally:
    conn.close()

TOPICOS_TEXTO = "\n".join(f"- {t}" for t in TOPICOS_NOMBRES)
print(f"Tópicos en BD: {len(TOPICOS_NOMBRES)}")
print(TOPICOS_TEXTO[:500], "..." if len(TOPICOS_TEXTO) > 500 else "")

Tópicos en BD: 8
- Limpieza
- Atención al cliente
- Instalaciones y servicios
- Relación calidad-precio
- Comodidad
- Desayuno / Gastronomía
- Ruido
- WiFi 


### Prompt y llamada al modelo

Misma salida JSON que el notebook CSV: `items[].(topico, fragmento, sentimiento_topico)`.

In [ ]:
SYSTEM_PROMPT = """Eres un analista de opiniones de hoteles. Tu tarea es leer una review en español y listar los TÓPICOS que menciona o implica.

Tópicos predefinidos (usa el nombre EXACTO si la review encaja):
{topicos}

Reglas:
- Una misma review puede tener VARIOS tópicos.
- Para cada tópico, copia un FRAGMENTO literal o casi literal de la review (cita breve) que justifique ese tópico.
- Para cada par tópico–fragmento, indica sentimiento_topico: la valoración que expresa ese fragmento respecto a ESE aspecto (no el sentimiento global de toda la review).
- Valores permitidos para sentimiento_topico: "positivo", "negativo" o "neutro". Usa "neutro" solo si el fragmento es factualmente equilibrado o no muestra clara satisfacción ni insatisfacción sobre ese aspecto.
- Si te damos un sentimiento global de referencia (p. ej. neutro), aun así debes asignar positivo/negativo/neutro por tópico según el fragmento; las reviews neutras suelen mezclar aspectos buenos y malos.
- Si el contenido no encaja bien en ninguno de los predefinidos, inventa un nombre breve y claro para el tópico (por ejemplo: "Ruido / descanso", "Ubicación", "Parking").
- No inventes fragmentos: deben aparecer en la review o ser un recorte mínimo fiel.
- Responde SOLO con un objeto JSON válido, sin markdown ni texto fuera del JSON.

Formato obligatorio:
{{"items": [{{"topico": "...", "slug": "...", "fragmento": "...", \
"sentimiento_topico": "sentimiento", "score_topico": valor entre 0.000 y 1.000}}, ...]}
""".format(topicos=TOPICOS_TEXTO)


def _normalizar_sentimiento_topico(val) -> str | None:
    if val is None or (isinstance(val, str) and not val.strip()):
        return None
    s = str(val).strip().lower()
    if s in ("positivo", "positive", "pos"):
        return "positivo"
    if s in ("negativo", "negative", "neg"):
        return "negativo"
    if s in ("neutro", "neutral"):
        return "neutro"
    return None


def parsear_respuesta_json(texto: str) -> list[dict]:
    texto = texto.strip()
    m = re.search(r"\{[\s\S]*\}", texto)
    if not m:
        return []
    blob = m.group(0)
    data = json.loads(blob)
    items = data.get("items", [])
    if not isinstance(items, list):
        return []
    out = []
    for it in items:
        if not isinstance(it, dict):
            continue
        top = str(it.get("topico", "")).strip()
        frag = str(it.get("fragmento", "")).strip()
        st = _normalizar_sentimiento_topico(it.get("sentimiento_topico"))
        if top and frag:
            out.append({"topico": top, "fragmento": frag, "sentimiento_topico": st})
    return out


def extraer_uso_tokens(resp: dict) -> dict:
    p = resp.get("prompt_eval_count")
    e = resp.get("eval_count")
    total = None
    if isinstance(p, int) and isinstance(e, int):
        total = p + e
    return {
        "prompt_eval_count": p,
        "eval_count": e,
        "total_tokens": total,
    }


def clasificar_review(
    texto_review: str,
    *,
    sentimiento_global: str | None = None,
    return_usage: bool = False,
):
    bloques = [f"Review:\n{texto_review}"]
    if sentimiento_global is not None and str(sentimiento_global).strip():
        bloques.append(
            "Sentimiento global etiquetado en el dataset (referencia, no sustituye el análisis por fragmento): "
            + str(sentimiento_global).strip()
        )
    user = "\n\n".join(bloques)
    r = ollama.chat(
        model=MODELO,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user},
        ],
        format="json",
        options={"temperature": 0.2},
    )
    content = r["message"]["content"]
    items = parsear_respuesta_json(content)
    if return_usage:
        return items, extraer_uso_tokens(r)
    return items

In [6]:
def _slug_disponible(cur, base: str) -> str:
    cand = base[:60]
    n = 0
    while True:
        cur.execute("SELECT 1 FROM topicos WHERE slug = %s", (cand,))
        if not cur.fetchone():
            return cand
        n += 1
        suf = f"_{n}"
        cand = (base[: 60 - len(suf)] + suf)[:60]


def obtener_o_crear_topico_id(cur, nombre_display: str, cache: dict[str, int]) -> int:
    """Resuelve nombre devuelto por el modelo a topicos.id; inserta fila nueva si hace falta."""
    nombre = (nombre_display or "").strip()
    if not nombre or nombre == "_error":
        raise ValueError("Nombre de tópico vacío o inválido")
    nk = _norm_nombre(nombre)
    if nk in cache:
        return cache[nk]
    cur.execute(
        "SELECT id FROM topicos WHERE lower(trim(nombre)) = lower(trim(%s))",
        (nombre,),
    )
    row = cur.fetchone()
    if row:
        cache[nk] = row[0]
        return row[0]
    base_slug = slugify(nombre) or "topico"
    slug = _slug_disponible(cur, base_slug)
    cur.execute(
        """
        INSERT INTO topicos (slug, nombre, tipo, umbral_alerta, activo)
        VALUES (%s, %s, 'adicional', 65, TRUE)
        RETURNING id
        """,
        (slug, nombre[:120]),
    )
    tid = cur.fetchone()[0]
    cache[nk] = tid
    return tid


UPSERT_REVIEW_TOPICO = """
INSERT INTO review_topicos (review_id, topico_id, fragmento, sentimiento, modelo_version)
VALUES (%s, %s, %s, %s, %s)
ON CONFLICT (review_id, topico_id) DO UPDATE SET
    fragmento = EXCLUDED.fragmento,
    sentimiento = EXCLUDED.sentimiento,
    modelo_version = EXCLUDED.modelo_version,
    procesado_en = NOW()
"""


def fetch_reviews_a_procesar(conn) -> list[tuple[int, str, str | None]]:
    """Lista de (review_id, texto_limpio, sentimiento_global opcional desde predicciones_sentimiento)."""
    filtros = []
    if ONLY_WITHOUT_TOPICOS:
        filtros.append(
            "NOT EXISTS (SELECT 1 FROM review_topicos rt WHERE rt.review_id = r.id)"
        )
    where_extra = " AND " + " AND ".join(filtros) if filtros else ""
    lim = ""
    params: list = []
    if N_MAX is not None:
        lim = "LIMIT %s OFFSET %s"
        params.extend([N_MAX, OFFSET])
    sql = f"""
    SELECT r.id, r.texto_limpio, ps.sentimiento
    FROM reviews r
    LEFT JOIN predicciones_sentimiento ps ON ps.review_id = r.id
    WHERE COALESCE(trim(r.texto_limpio), '') <> ''
    {where_extra}
    ORDER BY r.id
    {lim}
    """
    with conn.cursor() as cur:
        cur.execute(sql, params)
        return [(r[0], r[1], r[2]) for r in cur.fetchall()]

### Ollama en marcha (opcional en Windows)

In [7]:
import subprocess
import sys
import time
import socket


def iniciar_ollama():
    def is_ollama_running(host="localhost", port=11434):
        try:
            with socket.create_connection((host, port), timeout=1):
                return True
        except OSError:
            return False

    if not is_ollama_running():
        print("Ollama no está corriendo. Intentando iniciar...")
        try:
            if sys.platform.startswith("win"):
                subprocess.Popen(
                    "ollama serve",
                    shell=True,
                    creationflags=subprocess.DETACHED_PROCESS,
                )
            else:
                subprocess.Popen(
                    ["ollama", "serve"],
                    stdout=subprocess.DEVNULL,
                    stderr=subprocess.DEVNULL,
                )
            for _ in range(10):
                if is_ollama_running():
                    print("Ollama iniciado.")
                    break
                time.sleep(1)
            else:
                print("No se pudo iniciar Ollama automáticamente; inícialo manualmente.")
        except Exception as ex:
            print(f"Error al iniciar Ollama: {ex}")


iniciar_ollama()

### Prueba con una review de la BD

In [8]:
conn = conectar()
try:
    filas = fetch_reviews_a_procesar(conn)
finally:
    conn.close()

print("Reviews a procesar (según filtros):", len(filas))
if filas:
    rid, texto, sent = filas[0]
    items, uso = clasificar_review(texto, sentimiento_global=sent, return_usage=True)
    print("review_id:", rid)
    print(json.dumps(items, ensure_ascii=False, indent=2))
    print("Tokens:", uso)

Reviews a procesar (según filtros): 100
review_id: 605
[
  {
    "topico": "Instalaciones y servicios",
    "fragmento": "Tuvimos problemas con el aire acondicionado.",
    "sentimiento_topico": "negativo"
  }
]
Tokens: {'prompt_eval_count': 499, 'eval_count': 43, 'total_tokens': 542}


### Lote: predicción + `review_topicos` (+ `topicos` si aparece nombre nuevo)

Cada review se confirma en una transacción. Si una fila falla, se registra el error y se sigue.

In [9]:
try:
    from tqdm.auto import tqdm
except ImportError:

    def tqdm(x, **kwargs):
        return x


conn = conectar()
try:
    filas = fetch_reviews_a_procesar(conn)
finally:
    conn.close()

cache = dict(TOPICO_POR_NOMBRE_NORM)
ok = 0
err = []

for review_id, texto, sent in tqdm(filas, total=len(filas)):
    conn = conectar()
    try:
        try:
            items, _ = clasificar_review(
                texto, sentimiento_global=sent, return_usage=True
            )
        except Exception as e:
            err.append((review_id, f"ollama: {e}"))
            continue
        if not items:
            err.append((review_id, "sin items en JSON"))
            continue
        with conn:
            with conn.cursor() as cur:
                for it in items:
                    try:
                        tid = obtener_o_crear_topico_id(cur, it["topico"], cache)
                    except Exception as e:
                        err.append((review_id, f"tópico '{it.get('topico')}': {e}"))
                        continue
                    sent_t = _normalizar_sentimiento_topico(it.get("sentimiento_topico"))
                    cur.execute(
                        UPSERT_REVIEW_TOPICO,
                        (
                            review_id,
                            tid,
                            it["fragmento"],
                            sent_t,
                            MODEL_VERSION,
                        ),
                    )
        ok += 1
    except Exception as e:
        err.append((review_id, str(e)))
    finally:
        conn.close()

print(f"Reviews completadas: {ok} / {len(filas)} | errores: {len(err)}")
if err[:10]:
    print("Primeros errores:", err[:10])

  0%|          | 0/100 [00:00<?, ?it/s]

Reviews completadas: 100 / 100 | errores: 0


### Comprobación rápida en BD

In [10]:
conn = conectar()
try:
    with conn.cursor() as cur:
        cur.execute(
            """
            SELECT COUNT(*) AS n_review_topicos,
                   (SELECT COUNT(*) FROM topicos) AS n_topicos
            FROM review_topicos
            """
        )
        print(cur.fetchone())
        cur.execute(
            """
            SELECT rt.review_id, t.nombre, rt.sentimiento, left(rt.fragmento, 80)
            FROM review_topicos rt
            JOIN topicos t ON t.id = rt.topico_id
            ORDER BY rt.id DESC
            LIMIT 8
            """
        )
        for row in cur.fetchall():
            print(row)
finally:
    conn.close()

(111, 9)
(704, 'Limpieza', 'negativo', 'No había agua caliente en la ducha.')
(703, 'Instalaciones y servicios', 'negativo', 'Tuvimos problemas con el aire acondicionado.')
(702, 'Atención al cliente', 'negativo', 'No me gustó la atención en la recepción.')
(701, 'Instalaciones y servicios', 'positivo', 'El estacionamiento es amplio y seguro.')
(700, 'Instalaciones y servicios', 'negativo', 'Tuvimos problemas con el aire acondicionado.')
(699, 'Limpieza', 'negativo', 'El baño necesitaba mejor limpieza.')
(698, 'Atención al cliente', 'positivo', '[PERSONA] sin dudarlo, una gran experiencia.')
(697, 'Comodidad', 'positivo', 'era espaciosa')
